In [1]:
import math
from typing import List, Dict, Callable, Generator  # WYMÓG 8: Typowanie (type hinting)

# WYMÓG 4: Instrukcja pass (użyta do zdefiniowania pustego wyjątku biznesowego)
class DaneFinansoweError(Exception):
    pass


# WYMÓG 10: Zmienna liczba argumentów (*args do przyjmowania opcjonalnych odliczeń od dochodu)
def oblicz_skale(przychod: float, koszty: float, *odliczenia: float) -> float:
    dochod = max(0.0, przychod - koszty - sum(odliczenia))

    # WYMÓG 3a: Operator warunkowy potrójny (wyznaczenie podatku wg progów 12% i 32% dla 120 tys. zł)
    podatek_wstepny = dochod * 0.12 if dochod <= 120000 else (120000 * 0.12) + ((dochod - 120000) * 0.32)
    podatek = max(0.0, podatek_wstepny - 3600)  # Kwota zmniejszająca podatek 3600 zł

    # Składka zdrowotna 2026 dla skali: 9% dochodu, nie mniej niż 432.54 zł/mc (100% minimalnej)
    skladka_zdrowotna = max(432.54 * 12, dochod * 0.09)
    return podatek + skladka_zdrowotna


def oblicz_liniowy(przychod: float, koszty: float) -> float:
    dochod = max(0.0, przychod - koszty)

    # W 2026 r. składka zdrowotna liniowca to 4.9% dochodu (minimum 432.54 zł/mc). Max odliczenie 14100 zł.
    skladka_miesieczna = max(432.54, (dochod / 12) * 0.049)
    skladka_zdrowotna_roczna = skladka_miesieczna * 12

    dochod_opodatkowany = max(0.0, dochod - min(skladka_zdrowotna_roczna, 14100))
    podatek = dochod_opodatkowany * 0.19
    return podatek + skladka_zdrowotna_roczna


# WYMÓG 9: Argumenty domyślne i nazwane (stawka_ryczaltu ma wartość domyślną 12% — np. dla IT)
def oblicz_ryczalt(przychod: float, stawka_ryczaltu: float = 0.12) -> float:
    podatek = przychod * stawka_ryczaltu

    # Składka zdrowotna ryczałtu 2026 zależy od rocznego przychodu (progi: 60k i 300k zł)
    if przychod <= 60000:
        skladka_zdrowotna = 498.35 * 12
    elif przychod <= 300000:
        skladka_zdrowotna = 830.58 * 12
    else:
        skladka_zdrowotna = 1495.04 * 12

    return podatek + skladka_zdrowotna


# WYMÓG 7: Generator scenariuszy wzrostu przychodów (mnożniki: 100%, 125%, 150%)
def generator_scenariuszy(base_przychod: float) -> Generator[float, None, None]:
    for mnoznik in [1.0, 1.25, 1.5]:
        yield base_przychod * mnoznik


# GŁÓWNA FUNKCJA ANALITYCZNA
def uruchom_symulator() -> None:

    # Dane wejściowe firmy
    firma_info = "  PROFIL_FIRMY: Usługi Programistyczne B2B   "
    roczny_przychod = 160000.0
    roczne_koszty   = 20000.0

    # WYMÓG 1a: Funkcja przetwarzająca łańcuch znaków — .strip() (usunięcie białych znaków)
    # WYMÓG 1b: Funkcja przetwarzająca łańcuch znaków — .replace() (zamiana spacji na podkreślnik)
    profil_czysty = firma_info.strip().replace(" ", "_")

    # WYMÓG 6: Operator przynależności (in) — sprawdzenie słowa kluczowego w profilu działalności
    if "Programistyczne" in profil_czysty:
        stawka_rycz = 0.12   # IT: stawka 12%
    else:
        stawka_rycz = 0.15   # Pozostała działalność usługowa: 15%

    # WYMÓG 11: Funkcja anonimowa (lambda) do liczenia wyniku brutto przed podatkami
    oblicz_wynik_brutto = lambda p, k: p - k
    wynik_brutto = oblicz_wynik_brutto(roczny_przychod, roczne_koszty)

    # WYMÓG 5: Instrukcja match do wyboru odpowiedniej kalkulacji podatkowej
    formy_opodatkowania = ["Skala", "Liniowy", "Ryczałt"]
    wyniki: Dict[str, float] = {}

    for forma in formy_opodatkowania:
        match forma:
            case "Skala":
                # WYMÓG 10: Wywołanie z dodatkowym argumentem *args (ulga na Internet 760 zł)
                wyniki[forma] = oblicz_skale(roczny_przychod, roczne_koszty, 760.0)
            case "Liniowy":
                wyniki[forma] = oblicz_liniowy(roczny_przychod, roczne_koszty)
            case "Ryczałt":
                # WYMÓG 9: Wywołanie funkcji z użyciem argumentu nazwanego
                wyniki[forma] = oblicz_ryczalt(roczny_przychod, stawka_ryczaltu=stawka_rycz)

    # WYMÓG 2: Formatowanie łańcucha znaków (f-string z zaokrągleniem i wyrównaniem kolumn)
    print(f"\n{'=' * 65}")
    print(f"  Raport podatkowy 2026 dla: {profil_czysty}")
    print(f"{'=' * 65}")
    print(f"  Przychód roczny : {roczny_przychod:>10.2f} zł")
    print(f"  Koszty roczne   : {roczne_koszty:>10.2f} zł")
    print(f"  Wynik brutto    : {wynik_brutto:>10.2f} zł")
    print(f"{'-' * 65}")
    print(f"  {'Forma':<10} | {'Podatek + składka ZDR':>22} | {'Zysk netto':>12}")
    print(f"{'-' * 65}")

    for f, obciazenie in wyniki.items():
        czysty_zysk = wynik_brutto - obciazenie
        print(f"  {f:<10} | {obciazenie:>22.2f} zł | {czysty_zysk:>10.2f} zł")

    print(f"{'=' * 65}")

    # WYMÓG 3b: Operator Walrus (assignment expression) — najlepsza forma i rozpiętość różnicy
    najlepsza_forma = min(wyniki, key=wyniki.get)
    if (roznica := max(wyniki.values()) - min(wyniki.values())) > 0:
        print(f"\n  [Rekomendacja] Wybierz '{najlepsza_forma}'.")
        print(f"  Różnica między skrajnymi opcjami wynosi {roznica:.2f} zł rocznie.\n")

    # WYMÓG 7: Użycie generatora wprost — pobranie kolejnych scenariuszy przez next()
    print(f"{'-' * 65}")
    print(f"  Projekcja obciążeń przy wzroście przychodów ({najlepsza_forma}):")
    print(f"{'-' * 65}")

    gen = generator_scenariuszy(roczny_przychod)          # utworzenie generatora
    etykiety = ["Bazowy (100%)", "Wzrost  (+25%)", "Wzrost  (+50%)"]

    for etykieta in etykiety:
        scenariusz_przychod = next(gen)                   # WYMÓG 7: użycie next() na generatorze

        match najlepsza_forma:
            case "Skala":
                obciazenie_scen = oblicz_skale(scenariusz_przychod, roczne_koszty, 760.0)
            case "Liniowy":
                obciazenie_scen = oblicz_liniowy(scenariusz_przychod, roczne_koszty)
            case "Ryczałt":
                obciazenie_scen = oblicz_ryczalt(scenariusz_przychod, stawka_ryczaltu=stawka_rycz)
            case _:
                obciazenie_scen = 0.0

        zysk_scen = scenariusz_przychod - roczne_koszty - obciazenie_scen
        print(f"  {etykieta}: przychód {scenariusz_przychod:>9.0f} zł → zysk netto {zysk_scen:>9.2f} zł")

    print(f"{'=' * 65}\n")


# Uruchomienie programu
if __name__ == "__main__":
    uruchom_symulator()


  Raport podatkowy 2026 dla: PROFIL_FIRMY:_Usługi_Programistyczne_B2B
  Przychód roczny :  160000.00 zł
  Koszty roczne   :   20000.00 zł
  Wynik brutto    :  140000.00 zł
-----------------------------------------------------------------
  Forma      |  Podatek + składka ZDR |   Zysk netto
-----------------------------------------------------------------
  Skala      |               29488.40 zł |  110511.60 zł
  Liniowy    |               32156.60 zł |  107843.40 zł
  Ryczałt    |               29166.96 zł |  110833.04 zł

  [Rekomendacja] Wybierz 'Ryczałt'.
  Różnica między skrajnymi opcjami wynosi 2989.64 zł rocznie.

-----------------------------------------------------------------
  Projekcja obciążeń przy wzroście przychodów (Ryczałt):
-----------------------------------------------------------------
  Bazowy (100%): przychód    160000 zł → zysk netto 110833.04 zł
  Wzrost  (+25%): przychód    200000 zł → zysk netto 146033.04 zł
  Wzrost  (+50%): przychód    240000 zł → zysk nett